In [ ]:
print("\n" + "=" * 70)
print("STRATEGIC RECOMMENDATIONS")
print("=" * 70)

recommendations = f"""
1. RECOMMENDED MODEL: {best_model}
   - Best test R² = {best_test_r2:.4f}
   - Explains {best_test_r2*100:.1f}% of claim severity variance
   - Implementation: Use as core engine for dynamic pricing

2. PRICING STRATEGY
   - Current mean premium: Based on historical data
   - Optimized mean premium: R{optimized_df['OptimizedPremium'].mean():.2f}
   - Adjustment required: {optimized_df['PremiumAdjustmentPct'].mean():.1f}%
   - Rationale: ML-based risk quantification improves accuracy

3. RISK DRIVERS
   - Top features identified via SHAP analysis
   - Each feature's impact quantified and explainable
   - Regulatory compliance: All premium adjustments justified by data

4. MARGIN PROTECTION
   - Target margin: 20% on all policies
   - Expense ratio: 15% (operational costs)
   - Profit per policy: (Premium - Cost - Expenses)

5. NEXT STEPS
   - Implement A/B testing with segment of policies
   - Compare AI-optimized vs. traditional pricing
   - Measure: retention, loss ratio, profitability
   - Iterate model based on actual claims experience

6. RISK MITIGATION
   - Blend traditional and ML-based prices initially
   - Monitor edge cases (extreme age, high premium)
   - Quarterly model retraining with new claims data
"""

print(recommendations)

print("\n" + "=" * 70)
print(f"Analysis complete. Model ready for deployment.")
print("=" * 70)

## Section 8: Business Recommendations

Strategic insights from modeling analysis for pricing and risk management.

In [ ]:
print("=" * 70)
print("DYNAMIC PREMIUM OPTIMIZATION")
print("=" * 70)

# Optimize premiums using best model
optimized_df = optimize_premium(
    X_test.copy(),
    best_result,
    expense_ratio=0.15,  # 15% operating expenses
    profit_margin=0.20   # 20% profit margin
)

print(f"\nPremium optimization results:")
print(f"  Optimized premium - Mean: R{optimized_df['OptimizedPremium'].mean():.2f}")
print(f"  Optimized premium - Median: R{optimized_df['OptimizedPremium'].median():.2f}")
print(f"  Optimized premium - Std: R{optimized_df['OptimizedPremium'].std():.2f}")

# Premium adjustment analysis
print(f"\nPremium adjustment vs current:")
print(f"  Mean adjustment: R{optimized_df['PremiumAdjustment'].mean():.2f}")
print(f"  % adjustment (mean): {optimized_df['PremiumAdjustmentPct'].mean():.2f}%")
print(f"  % adjustment (median): {optimized_df['PremiumAdjustmentPct'].median():.2f}%")

print(f"\nBreakdown of premium components:")
print(f"  Pure Premium (Loss): R{optimized_df['PurePremium'].mean():.2f}")
print(f"  Expense Loading (15%): R{optimized_df['ExpenseLoading'].mean():.2f}")
print(f"  Final Premium with Margin: R{optimized_df['OptimizedPremium'].mean():.2f}")

## Section 7: Dynamic Premium Optimization

Generate risk-based premiums using model predictions and profit targets.

In [ ]:
print("=" * 70)
print("SHAP FEATURE INTERPRETABILITY")
print("=" * 70)

try:
    # Calculate SHAP values for best model (sample to avoid memory issues)
    sample_size = min(100, len(X_test))
    X_sample = X_test.iloc[:sample_size]
    
    print(f"Calculating SHAP values for {sample_size} test samples...")
    shap_result = calculate_shap_values(best_result, X_sample, max_samples=sample_size)
    
    print(f"✓ SHAP values calculated successfully")
    print(f"  Feature importance (from SHAP):")
    for i, imp in enumerate(np.sort(shap_result['feature_importance'])[::-1][:5]):
        print(f"    Top {i+1}: {imp:.4f}")
    
    print(f"\nTop risk drivers identified. Use SHAP plots in actual analysis for visualization.")
    
except Exception as e:
    print(f"Note: SHAP analysis requires tree-based models. Error: {str(e)[:50]}")

## Section 6: SHAP Interpretability Analysis

Generate SHAP values to explain model predictions and top risk drivers.

In [ ]:
print("=" * 70)
print("FEATURE IMPORTANCE")
print("=" * 70)

# Get feature importance from best model
if best_model == 'Random Forest':
    best_result = rf_result
elif best_model == 'XGBoost':
    best_result = xgb_result
else:
    best_result = lr_result

if 'feature_importance' in best_result:
    feature_importance = best_result['feature_importance']
    print(f"Feature importance (Top 10 from {best_model}):")
    print(f"  Mean importance score: {feature_importance.mean():.4f}")
    print(f"  Max importance: {feature_importance.max():.4f}")
    print(f"  Total importance sum: {feature_importance.sum():.4f}")

print("\nNote: Use SHAP analysis below for detailed feature interpretation.")

## Section 5: Feature Importance Analysis

Identify the most influential features driving claim predictions.

In [ ]:
# Compare models
print("\n" + "=" * 70)
print("MODEL COMPARISON")
print("=" * 70)

comparison_df = compare_models([lr_result, rf_result, xgb_result])
print(comparison_df.to_string(index=False))

# Identify best model
best_test_r2 = comparison_df['Test R²'].max()
best_model_idx = comparison_df['Test R²'].idxmax()
best_model = comparison_df.iloc[best_model_idx]['Model']

print(f"\n✓ Best Model: {best_model} (Test R² = {best_test_r2:.4f})")

## Section 4: Model Comparison

Compare performance across all three algorithms.

In [ ]:
print("Training models...")
print("=" * 70)

# 1. Linear Regression
lr_result = train_linear_regression(X_train, y_train, X_test, y_test)
print(format_model_summary(lr_result))

# 2. Random Forest
rf_result = train_random_forest(X_train, y_train, X_test, y_test, 
                                n_estimators=100, max_depth=15)
print(format_model_summary(rf_result))

# 3. XGBoost
xgb_result = train_xgboost(X_train, y_train, X_test, y_test,
                           n_estimators=100, max_depth=6)
print(format_model_summary(xgb_result))

## Section 3: Model Training

Train three machine learning models for claim severity prediction.

In [ ]:
# Prepare features for modeling
X, y = prepare_features(df, target_column='TotalClaims')

print(f"Features prepared: {X.shape[1]} features from {len(X)} records")
print(f"\nFeature types:")
print(f"  Numerical: {len(X.select_dtypes(include=['int64', 'float64']).columns)}")
print(f"  Categorical: {len(X.select_dtypes(include=['object']).columns)}")

# Train/test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print(f"\nTrain/Test Split:")
print(f"  Train: {len(X_train)} records")
print(f"  Test: {len(X_test)} records")
print(f"  Target statistics:")
print(f"    Train - Mean: R{y_train.mean():.2f}, Std: R{y_train.std():.2f}")
print(f"    Test - Mean: R{y_test.mean():.2f}, Std: R{y_test.std():.2f}")

## Section 2: Feature Engineering

Prepare features for modeling with domain-informed transformations.

In [ ]:
# Load and prepare data
df = load_data('../data/insurance_data_cleaned.csv')
df = calculate_loss_metrics(df)

print(f"Dataset loaded: {len(df)} records, {len(df.columns)} columns")
print(f"Date range: {df['TransactionMonth'].min()} to {df['TransactionMonth'].max()}")
print(f"\nTarget statistics:")
print(f"  TotalClaims - Mean: R{df['TotalClaims'].mean():.2f}, Median: R{df['TotalClaims'].median():.2f}")
print(f"  TotalClaims - Std Dev: R{df['TotalClaims'].std():.2f}, Max: R{df['TotalClaims'].max():.2f}")

# Summary statistics
df.describe()

## Section 1: Data Loading and Preparation

Load cleaned insurance data and engineer features for modeling.

In [ ]:
import sys
sys.path.append('../')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error

from src.data_loader import load_data
from src.eda_utils import calculate_loss_metrics
from src.modeling import (
    prepare_features,
    train_linear_regression,
    train_random_forest,
    train_xgboost,
    compare_models,
    calculate_shap_values,
    optimize_premium,
    format_model_summary
)

# Configure display
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)
sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (12, 6)

# Insurance Risk Modeling & Dynamic Pricing System

Build and evaluate machine learning models for claim severity prediction and dynamic, risk-based premium optimization for ACIS.

**Modeling Approach:**
- **Claim Severity Model**: Predict TotalClaims amount to estimate financial liability
- **Premium Optimization**: Dynamic pricing using P(claim) × Predicted Severity + Expenses + Margin
- **Algorithms**: Linear Regression (baseline), Random Forest (ensemble), XGBoost (gradient boosting)
- **Interpretability**: SHAP analysis of top risk drivers for pricing rationale

**Expected Outcomes:**
- Improved pricing accuracy through ML-based risk quantification
- Regulatory justification through explainable features (SHAP values)
- Premium optimization framework for competitive advantage